# 02 · Data Modeling
**Goal:** Decide how the four tables relate to each other, validate that the join keys actually work, and design the schema of the master synthetic dataset before cleaning or merging

### 1. Load the four raw datasets
Same four datasets as the data_audit notebook.

In [3]:
import pandas as pd

menu         = pd.read_csv('../data/raw/starbucks_menu.csv')
transactions = pd.read_csv('../data/raw/synthetic_transactions.csv')
macro        = pd.read_csv('../data/raw/fred_macro.csv')
weather      = pd.read_csv('../data/raw/weather_daily.csv')

### 2. Star schema

From previous results in data audit, TRANSACTION is the largest and most granular table among the four. The other three tables describes something a transaction references (what was bought - MENU, what the weather was - WEATHER, what the economy looked like that month - MACRO).

Therefore, TRANSACTION behaves as the **fact table**, and MENU, WEATHER, MACRO are the **dimension tables**.

The fact table and dimension tables together form the Star Schema.

In [4]:
for name, df in [('menu', menu), ('transactions', transactions), ('macro', macro), ('weather', weather)]:
    print(f'{name:15s} {df.shape[0]:>7,} rows  {df.shape[1]:>2} cols')

menu                242 rows   8 cols
transactions    100,000 rows  20 cols
macro                60 rows   4 cols
weather           3,655 rows   5 cols


### 3. Validating the join keys

**`(product_name, size)`: Menu vs. Transaction**

In [5]:
def keyset(df, cols):
    return set(df[cols].drop_duplicates().itertuples(index=False, name=None))

menu_keys = keyset(menu, ['product_name', 'size'])
tx_keys   = keyset(transactions, ['product_name', 'size'])

print('menu unique (product_name, size):       ', len(menu_keys))
print('transaction unique (product_name, size):', len(tx_keys))
print('transaction keys missing from menu:     ', tx_keys - menu_keys)

menu unique (product_name, size):        106
transaction unique (product_name, size): 106
transaction keys missing from menu:      set()


MENU and TRANSACTION tables both have 106 sets of unique (product_name, size), and since their difference set is the an empty set, this means every (product_name, size) in TRANSACTION exactly matches one (product_name, size) in MENU.

Therefore, if we use MENU as a dimension table to JOIN TRANSACTION, there will not exists any circumstances in which the join fails or NULL values.

**`category` and `city`**

In [6]:
print('category values in transactions but not menu:', set(transactions['category']) - set(menu['category']))
print('city values in transactions but not weather:  ', set(transactions['city']) - set(weather['city']))

category values in transactions but not menu: set()
city values in transactions but not weather:   set()


Every (product_name, size) combination, category, and city in TRANSACTION has a match on the dimension tables (MENU and WEATHER). That confirms that it is safe to build WEATHER to TRANSACTION, as well as category/city relationships.

### 4. The problem: Menu is not one row per product

In [7]:
print('menu shape:', menu.shape)
print('unique (product_name, size) combinations:', len(menu_keys))

menu shape: (242, 8)
unique (product_name, size) combinations: 106


242 rows for 106 keys means most products (name-size combinations) appear more than once. That could just be duplicate rows, but checking whether the rows with shared products have the same values is important. If a (product_name, size) in TRANSACTION could match several (product_name, size) in MENU with different values, this will lead to fan-out (when we have several choices to match a product, but no idea of using witch value).

In [10]:
dup_groups = menu.groupby(['product_name', 'size'])

n_varying = 0

for key, group in dup_groups:
    if len(group) <= 1:
        continue

    values_only = group.drop(columns=['product_name', 'size'])
    unique_counts = values_only.nunique()
    has_conflict = (unique_counts > 1).any()

    if has_conflict:
        n_varying += 1

print(f'{n_varying} of {dup_groups.ngroups} product/size groups have rows with genuinely different calories/price/sugar/caffeine values')
#Print below

menu[(menu['product_name'] == 'Caffè Latte') & (menu['size'] == 'Grande')]
#A example product/size combination that have rows with different values. Shown as the table below. While all 9 rows are all (Caffè Latte, Grande), each of theirs price_usd, calories, sugar_g, caffeine_mg are different.

24 of 106 product/size groups have rows with genuinely different calories/price/sugar/caffeine values


,product_name,category,size,price_usd,calories,sugar_g,caffeine_mg,seasonal_flag
5,Caffè Latte,Classic Espresso Drinks,Grande,4.21,100,9,75.0,0
6,Caffè Latte,Classic Espresso Drinks,Grande,4.49,70,4,75.0,0
8,Caffè Latte,Classic Espresso Drinks,Grande,4.18,150,14,75.0,0
9,Caffè Latte,Classic Espresso Drinks,Grande,4.33,110,6,75.0,0
10,Caffè Latte,Classic Espresso Drinks,Grande,4.18,130,18,150.0,0
11,Caffè Latte,Classic Espresso Drinks,Grande,4.18,190,17,150.0,0
12,Caffè Latte,Classic Espresso Drinks,Grande,4.29,150,8,150.0,0
14,Caffè Latte,Classic Espresso Drinks,Grande,3.99,240,22,150.0,0
15,Caffè Latte,Classic Espresso Drinks,Grande,4.17,190,11,150.0,0


Therefore, this is a real data quality problem. For example, the Grande Caffè Latte alone shows up with prices from \$3.99 to \$4.49 and calories from 70 to 240.

**Decision:** Menu will be deduplicated to one row per (product_name, size) in `03_data_cleaning.ipynb` before it's used as a dimension table.

However, since TRANSACTION table already has its own base_price, calories, sugar_g, caffeine_mg columns, the only field the master dataset actually needs to add from the cleaned MENU dimension table is seasonal_flag.

Therefore, the data quality problem of MENU described above have no effect on our JOIN of MENU and TRANSACTION tables.

### 5. Grain mismatch: Macro is monthly, but Transaction is daily

MACRO has only one row per calendar month; TRANSACTION has one row per sale.

Joining them has to first truncating Transaction.date down to a month. Once that's done, does every transaction month have a matching macro row?

In [9]:
transactions['month'] = pd.to_datetime(transactions['date']).dt.to_period('M').astype(str)
macro['month'] = pd.to_datetime(macro['date']).dt.to_period('M').astype(str)

missing_months = set(transactions['month']) - set(macro['month'])
print('transaction months with no macro coverage:', missing_months)
print('affected transactions:', transactions['month'].isin(missing_months).sum())

transaction months with no macro coverage: {'2026-03'}
affected transactions: 122


MACRO data ends at 2026-02-01, but TRANSACTION has rows in 2026-03. Those rows will get a NULL after joining MACRO to TRANSACTION and 122 transactions will be affected.

How to handle it (leave NULL, forward-fill the last known CPI, or drop those rows?) is a cleaning decision, and will be detailed display in `03_data_cleaning.ipynb`.

### 6. Findings summary

| Relationship | Status | Action |
|---|---|---|
| Transaction.(product_name, size) → Menu.(product_name, size) | Keys match, but Menu has duplicate/conflicting rows per key | Dedupe Menu in Phase 3 |
| Transaction.category → Menu.category | Clean, 1:1 | None needed |
| Transaction.city → Weather.city | Clean, 1:1 | None needed |
| Transaction.(date, city) → Weather.(date, city) | Clean, dates already overlap in range | None needed |
| Transaction.month → Macro.month | Clean except 2026-03 (122 rows, no macro coverage) | Decide null-handling strategy in Phase 3 |